# Análise Exploratória e Diagnóstico de Qualidade (EDA)

**Problema de negócio:** Quais fatores explicam atrasos de entrega e quedas de satisfação — e é possível prever ambos antes que aconteçam?

**Dois alvos de modelagem:**
- `flag_atraso` — `order_delivered_customer_date > order_estimated_delivery_date`
- `flag_review_ruim` — `review_score <= 3`

Este notebook mapeia inconsistências, valores nulos e duplicatas nas tabelas Raw do Olist. As conclusões de cada seção fundamentam as regras de limpeza da **Camada Staging** e a seleção de features dos modelos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

# Carrega os datasets
paths = glob.glob("../data/raw/*.csv")
dfs = {os.path.basename(p).replace(".csv", ""): pd.read_csv(p) for p in paths}
print(f"{len(dfs)} tabelas carregadas:")
for nome, df in dfs.items():
    print(f"  {nome}: {df.shape}")

## 1. Visão Geral: Nulos e Duplicatas
Um loop por todas as tabelas para mapear o cenário macro.

In [2]:
for nome, df in dfs.items():
    print(f"\n=== {nome} | shape={df.shape} ===")
    nulos = (df.isnull().mean() * 100).round(2)
    print("Nulos (%):")
    print(nulos[nulos > 0].to_dict() if (nulos > 0).any() else "{}")
    print("Duplicatas:", df.duplicated().sum())


=== product_category_name_translation | shape=(71, 2) ===
Nulos (%):
{}
Duplicatas: 0

=== olist_products_dataset | shape=(32951, 9) ===
Nulos (%):
{'product_category_name': 1.85, 'product_name_lenght': 1.85, 'product_description_lenght': 1.85, 'product_photos_qty': 1.85, 'product_weight_g': 0.01, 'product_length_cm': 0.01, 'product_height_cm': 0.01, 'product_width_cm': 0.01}
Duplicatas: 0

=== olist_order_payments_dataset | shape=(103886, 5) ===
Nulos (%):
{}
Duplicatas: 0

=== olist_order_reviews_dataset | shape=(99224, 7) ===
Nulos (%):
{'review_comment_title': 88.34, 'review_comment_message': 58.7}
Duplicatas: 0

=== olist_order_items_dataset | shape=(112650, 7) ===
Nulos (%):
{}
Duplicatas: 0

=== olist_customers_dataset | shape=(99441, 5) ===
Nulos (%):
{}
Duplicatas: 0

=== olist_sellers_dataset | shape=(3095, 4) ===
Nulos (%):
{}
Duplicatas: 0

=== olist_orders_dataset | shape=(99441, 8) ===
Nulos (%):
{'order_approved_at': 0.16, 'order_delivered_carrier_date': 1.79, 'order_de

## 2. Diagnóstico: Pedidos (`olist_orders_dataset`)
Existem nulos em datas importantes de entrega. Como nosso projeto foca em **predição de atraso**, não podemos usar pedidos sem data de entrega.

In [3]:
orders = dfs["olist_orders_dataset"]
# Filtrar apenas pedidos Entregues
entregues = orders[orders["order_status"] == "delivered"]
print(f"Total de pedidos originais: {len(orders)}")
print(f"Total de pedidos entregues: {len(entregues)}")
print(f"Nulos na data de entrega ao cliente (entre os Entregues): {entregues['order_delivered_customer_date'].isnull().sum()}")

Total de pedidos originais: 99441
Total de pedidos entregues: 96478
Nulos na data de entrega ao cliente (entre os Entregues): 8


> **Conclusão:** Em Analytics, usaremos um filtro (`WHERE order_status = 'delivered' AND delivered_ts IS NOT NULL`).

## 3. Diagnóstico: Avaliações (`olist_order_reviews_dataset`)
A grande maioria não deixa comentários. E precisamos verificar a cardinalidade (1 pedido = 1 review?).

In [4]:
reviews = dfs["olist_order_reviews_dataset"]
print("Total de avaliações:", len(reviews))
print("Pedidos únicos com avaliação:", reviews["order_id"].nunique())
print(f"Pedidos com mais de uma avaliação: {len(reviews) - reviews['order_id'].nunique()}")

Total de avaliações: 99224
Pedidos únicos com avaliação: 98673
Pedidos com mais de uma avaliação: 551


> **Conclusão:** Existem pedidos com mais de um review. Na Staging, usaremos `DISTINCT ON (order_id)` pegando a mais recente para manter o grão de 1 linha = 1 pedido.

## 4. Diagnóstico: Geolocalização (`olist_geolocation_dataset`)
É a tabela com maior risco de causar explosão de linhas nos Joins (Produto Cartesiano).

In [ ]:
geo = dfs["olist_geolocation_dataset"]
print("Total de linhas:", len(geo))
print("Duplicatas exatas:", geo.duplicated().sum())
print("Prefixos de CEP únicos:", geo["geolocation_zip_code_prefix"].nunique())

> **Conclusão:** Temos mais de 1 milhão de linhas, mas apenas ~19 mil CEPs únicos. Na Staging, precisamos agrupar (`GROUP BY`) por prefixo de CEP e tirar a média das coordenadas para desduplicar.

## 5. Análise Temporal

Antes de construir qualquer modelo, precisamos entender a janela temporal dos dados. Pedidos de 2016 são pouquíssimos (dataset incompleto) e podem distorcer análises sazonais e splits de treino/teste.

In [ ]:
orders = dfs["olist_orders_dataset"].copy()
orders["purchase_ts"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["ano_mes"] = orders["purchase_ts"].dt.to_period("M")
orders["ano"] = orders["purchase_ts"].dt.year

por_ano = orders["ano"].value_counts().sort_index()
print("Pedidos por ano:")
print(por_ano.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Volume por mês
mensal = orders.groupby("ano_mes").size()
mensal.plot(ax=axes[0], title="Volume de Pedidos por Mês", color="steelblue")
axes[0].set_xlabel("Mês"); axes[0].set_ylabel("Qtd. Pedidos")

# Pizza por ano
por_ano.plot.pie(ax=axes[1], autopct="%1.1f%%", title="Distribuição por Ano",
                 colors=["#d9534f", "#5bc0de", "#5cb85c"])
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../reports/temporal_distribuicao.png", bbox_inches="tight")
plt.show()

> **Conclusão:** 2016 tem volume desprezível (< 1% dos pedidos) e representa um período de lançamento do marketplace, não o comportamento maduro. Na Staging e nos modelos, filtraremos `purchase_ts >= '2017-01-01'`.

## 6. Distribuição de Status dos Pedidos

Nosso modelo de atraso só faz sentido para pedidos **entregues** — é o único status onde temos `order_delivered_customer_date` preenchido para calcular o delta em relação ao prazo estimado.

In [ ]:
status_dist = orders["order_status"].value_counts()
pct_delivered = status_dist["delivered"] / len(orders) * 100
print(f"Distribuição de status:\n{status_dist.to_string()}")
print(f"\nPedidos 'delivered': {status_dist['delivered']:,} ({pct_delivered:.1f}% do total)")

fig, ax = plt.subplots(figsize=(9, 4))
status_dist.plot.barh(ax=ax, color="steelblue")
ax.set_title("Distribuição de Status dos Pedidos")
ax.set_xlabel("Quantidade")
ax.bar_label(ax.containers[0], padding=4)
plt.tight_layout()
plt.savefig("../reports/status_distribuicao.png", bbox_inches="tight")
plt.show()

> **Conclusão:** ~97% dos pedidos têm status `delivered`. Os demais (cancelados, em processamento etc.) serão excluídos da análise de atraso por ausência da data de entrega real. O filtro `WHERE order_status = 'delivered' AND delivered_ts IS NOT NULL` preserva a grande maioria da base.

## 7. Distribuição do Review Score

O review score é nossa segunda variável-alvo (`flag_review_ruim = score <= 3`). Precisamos entender a distribuição para antecipar o **desbalanceamento de classes** — se notas boas (4–5) forem a grande maioria, o modelo precisará de estratégias compensatórias (`class_weight`, `stratify` no split, métricas de F1/AUC).

In [ ]:
reviews = dfs["olist_order_reviews_dataset"].copy()

score_dist = reviews["review_score"].value_counts().sort_index()
pct_ruim = (reviews["review_score"] <= 3).mean() * 100
print(f"Distribuição de review_score:\n{score_dist.to_string()}")
print(f"\nReviews com score <= 3 (flag_review_ruim=1): {pct_ruim:.1f}%")
print(f"Reviews com score >= 4 (flag_review_ruim=0): {100-pct_ruim:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ["#d9534f" if s <= 3 else "#5cb85c" for s in score_dist.index]
score_dist.plot.bar(ax=axes[0], color=colors, edgecolor="white")
axes[0].set_title("Distribuição de Review Score")
axes[0].set_xlabel("Score"); axes[0].set_ylabel("Qtd.")
axes[0].set_xticklabels(score_dist.index, rotation=0)
axes[0].bar_label(axes[0].containers[0], padding=3)
axes[0].legend(handles=[
    plt.Rectangle((0,0),1,1, color="#d9534f", label="Ruim (≤3)"),
    plt.Rectangle((0,0),1,1, color="#5cb85c", label="Bom (≥4)")
], loc="upper left")

flag_dist = pd.Series({"Bom (score ≥ 4)": 100 - pct_ruim, "Ruim (score ≤ 3)": pct_ruim})
flag_dist.plot.pie(ax=axes[1], autopct="%1.1f%%",
                   colors=["#5cb85c", "#d9534f"], startangle=90)
axes[1].set_title("flag_review_ruim — Desbalanceamento")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../reports/review_score_distribuicao.png", bbox_inches="tight")
plt.show()

> **Conclusão:** A distribuição é fortemente assimétrica à direita (pico em 5). `flag_review_ruim` será a classe minoritária (~25–30%), o que exige `class_weight="balanced"` nos modelos e uso de F1/AUC como métricas principais — acurácia bruta será enganosa.

## 8. Definição e Diagnóstico das Variáveis-Alvo

Esta é a seção mais crítica do EDA: calculamos as duas variáveis-alvo diretamente dos dados brutos para validar sua viabilidade e entender o desbalanceamento real antes de qualquer ETL.

In [ ]:
orders_ts = dfs["olist_orders_dataset"].copy()
for col in ["order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders_ts[col] = pd.to_datetime(orders_ts[col])

# Base de trabalho: apenas entregues com datas completas
entregues = orders_ts[
    (orders_ts["order_status"] == "delivered") &
    orders_ts["order_delivered_customer_date"].notna() &
    orders_ts["order_estimated_delivery_date"].notna()
].copy()

entregues["atraso_dias"] = (
    entregues["order_delivered_customer_date"] - entregues["order_estimated_delivery_date"]
).dt.days
entregues["flag_atraso"] = (entregues["atraso_dias"] > 0).astype(int)

taxa_atraso = entregues["flag_atraso"].mean() * 100
print(f"Base entregues (com datas completas): {len(entregues):,} pedidos")
print(f"\n--- Alvo 1: flag_atraso ---")
print(f"  Atrasados (flag=1): {entregues['flag_atraso'].sum():,} ({taxa_atraso:.1f}%)")
print(f"  No prazo  (flag=0): {(entregues['flag_atraso']==0).sum():,} ({100-taxa_atraso:.1f}%)")
print(f"\n  Estatísticas de atraso_dias (entre os ATRASADOS):")
print(entregues.loc[entregues["flag_atraso"]==1, "atraso_dias"].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

entregues["atraso_dias"].clip(-30, 60).plot.hist(
    ax=axes[0], bins=60, color="steelblue", edgecolor="white"
)
axes[0].axvline(0, color="red", linestyle="--", label="Limite (0 dias)")
axes[0].set_title("Distribuição de Atraso em Dias\n(atraso_dias = entregue − estimado)")
axes[0].set_xlabel("Dias"); axes[0].set_ylabel("Freq.")
axes[0].legend()

pd.Series({"No prazo (flag=0)": 100-taxa_atraso, "Atrasado (flag=1)": taxa_atraso}).plot.pie(
    ax=axes[1], autopct="%1.1f%%", colors=["#5cb85c", "#d9534f"], startangle=90
)
axes[1].set_title("flag_atraso — Desbalanceamento")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("../reports/flag_atraso_diagnostico.png", bbox_inches="tight")
plt.show()

> **Conclusão:** A maioria dos pedidos é entregue antes do prazo (valores negativos de `atraso_dias`). `flag_atraso` é a classe minoritária (~8–10%), exigindo as mesmas estratégias de balanceamento que `flag_review_ruim`. Atenção: valores de `atraso_dias` muito negativos (entrega muito antes do prazo) indicam que a Olist tem margem de segurança generosa nas estimativas.

## 9. Conexão entre Atraso e Satisfação

Uma das hipóteses centrais do projeto: pedidos atrasados geram notas mais baixas. Aqui validamos visualmente essa relação ainda nos dados brutos — o que motiva o uso de `flag_atraso` como feature no Modelo 2.

In [ ]:
reviews_raw = dfs["olist_order_reviews_dataset"][["order_id", "review_score"]].copy()

# Pega o review mais recente por pedido (mesma lógica da Staging)
reviews_dedup = (
    reviews_raw.sort_values("review_score")
    .drop_duplicates(subset="order_id", keep="last")
)

analise = entregues[["order_id", "flag_atraso", "atraso_dias"]].merge(
    reviews_dedup, on="order_id", how="inner"
)
analise["flag_review_ruim"] = (analise["review_score"] <= 3).astype(int)

print(f"Pedidos com atraso E review: {len(analise):,}")
print("\nNota média por situação de entrega:")
print(analise.groupby("flag_atraso")["review_score"].agg(["mean", "median", "count"]).round(2))
print(f"\nTaxa de review ruim — No prazo: {analise.loc[analise.flag_atraso==0,'flag_review_ruim'].mean()*100:.1f}%")
print(f"Taxa de review ruim — Atrasado: {analise.loc[analise.flag_atraso==1,'flag_review_ruim'].mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot nota por situação
analise.boxplot(column="review_score", by="flag_atraso", ax=axes[0],
                patch_artist=True,
                boxprops=dict(facecolor="steelblue", color="navy"),
                medianprops=dict(color="red", linewidth=2))
axes[0].set_title("Review Score por Situação de Entrega")
axes[0].set_xlabel("flag_atraso (0=no prazo, 1=atrasado)")
axes[0].set_ylabel("Review Score")
plt.sca(axes[0]); plt.title("Review Score por Situação de Entrega")

# Distribuição de score por grupo
for flag, grupo in analise.groupby("flag_atraso"):
    label = "Atrasado" if flag == 1 else "No prazo"
    grupo["review_score"].value_counts(normalize=True).sort_index().plot(
        kind="bar", ax=axes[1], alpha=0.65, label=label,
        color="#d9534f" if flag == 1 else "#5cb85c", edgecolor="white"
    )
axes[1].set_title("% de cada Score por Situação")
axes[1].set_xlabel("Score"); axes[1].set_ylabel("Proporção")
axes[1].legend(); axes[1].set_xticklabels([1,2,3,4,5], rotation=0)

plt.tight_layout()
plt.savefig("../reports/atraso_vs_review.png", bbox_inches="tight")
plt.show()

> **Conclusão:** Pedidos atrasados têm nota média e mediana visivelmente menores, e taxa de review ruim muito superior. Isso confirma que `flag_atraso` é um **preditor forte** de `flag_review_ruim` e que os dois modelos estão conceitualmente encadeados. O teste de hipótese formal (Mann-Whitney U) será conduzido no `02_estatistica.ipynb`.

## 10. Mapa de Relacionamentos entre Tabelas

Antes de construir a camada Analytics, é fundamental ter clara a cardinalidade de cada junção para evitar o **produto cartesiano** — o principal risco de explosão de linhas em pipelines SQL.

In [ ]:
orders_df    = dfs["olist_orders_dataset"]
items_df     = dfs["olist_order_items_dataset"]
payments_df  = dfs["olist_order_payments_dataset"]
reviews_df   = dfs["olist_order_reviews_dataset"]
customers_df = dfs["olist_customers_dataset"]
products_df  = dfs["olist_products_dataset"]
sellers_df   = dfs["olist_sellers_dataset"]

print("=== Cardinalidade das Junções ===\n")

# orders → customers (1:1 esperado via customer_id)
merged_cust = orders_df.merge(customers_df, on="customer_id", how="left")
print(f"orders JOIN customers via customer_id:")
print(f"  orders: {len(orders_df):,} | após join: {len(merged_cust):,} {'✓ 1:1' if len(merged_cust)==len(orders_df) else '⚠ expansão!'}\n")

# orders → order_items (1:N)
itens_por_pedido = items_df.groupby("order_id").size()
print(f"orders JOIN order_items via order_id (1:N):")
print(f"  Itens por pedido — min:{itens_por_pedido.min()} | max:{itens_por_pedido.max()} | mediana:{itens_por_pedido.median()}")
print(f"  Pedidos com >1 item: {(itens_por_pedido > 1).sum():,} ({(itens_por_pedido > 1).mean()*100:.1f}%)\n")

# orders → order_payments (1:N)
pag_por_pedido = payments_df.groupby("order_id").size()
print(f"orders JOIN order_payments via order_id (1:N):")
print(f"  Pagamentos por pedido — min:{pag_por_pedido.min()} | max:{pag_por_pedido.max()} | mediana:{pag_por_pedido.median()}")
print(f"  Pedidos com >1 pagamento: {(pag_por_pedido > 1).sum():,} ({(pag_por_pedido > 1).mean()*100:.1f}%)\n")

# orders → order_reviews (1:N na prática, queremos 1:1)
rev_por_pedido = reviews_df.groupby("order_id").size()
print(f"orders JOIN order_reviews via order_id:")
print(f"  Reviews por pedido — min:{rev_por_pedido.min()} | max:{rev_por_pedido.max()}")
print(f"  Pedidos com >1 review: {(rev_por_pedido > 1).sum():,} → requer DISTINCT ON na Staging\n")

# products → category translation
products_df2 = dfs["olist_products_dataset"]
cat_trans    = dfs["product_category_name_translation"]
cats_sem_trad = set(products_df2["product_category_name"].dropna()) - set(cat_trans["product_category_name"])
print(f"Categorias de produtos sem tradução: {len(cats_sem_trad)}")
if cats_sem_trad:
    print(f"  Exemplos: {list(cats_sem_trad)[:5]}")

> **Conclusão:** `orders` é o hub central. Antes de qualquer `JOIN`, itens e pagamentos precisam ser **agregados** por `order_id` (SUM/COUNT), e reviews precisam de **deduplicação** (DISTINCT ON). Fazer o join direto sem agregação prévia causa produto cartesiano e duplica linhas da fato.

## 11. Consolidação — Regras para a Camada Staging

Resumo de todas as decisões de limpeza derivadas deste EDA, organizadas por tabela.

| Tabela | Problema identificado | Regra na Staging |
|---|---|---|
| `olist_orders_dataset` | Nulos em datas de entrega; pedidos não entregues; dados 2016 incompletos | `WHERE order_status = 'delivered' AND delivered_ts IS NOT NULL AND purchase_ts >= '2017-01-01'` |
| `olist_order_reviews_dataset` | 551 pedidos com >1 review | `DISTINCT ON (order_id) ORDER BY review_creation_date DESC` |
| `olist_geolocation_dataset` | 261k duplicatas exatas; múltiplos registros por CEP | `GROUP BY zip_prefix` com `AVG(lat), AVG(lng)` |
| `olist_products_dataset` | 1.85% nulos em categoria e dimensões | Preencher categoria nula com `'unknown'`; imputar mediana nos campos numéricos |
| `olist_order_items_dataset` | 1:N com orders | Agregar antes do JOIN: `SUM(price)`, `SUM(freight_value)`, `COUNT(*)`, `COUNT(DISTINCT seller_id)` |
| `olist_order_payments_dataset` | 1:N com orders | Agregar: `SUM(payment_value)`, `MAX(payment_installments)`, `mode(payment_type)` |

**Decisões sobre as variáveis-alvo:**
- `flag_atraso = 1` se `delivered_ts > estimated_ts`, classe minoritária (~8–10%)
- `flag_review_ruim = 1` se `review_score <= 3`, classe minoritária (~25–30%)
- Ambos exigem `class_weight="balanced"` e métricas de F1/AUC nos modelos